# Total Energy

Calculate the total energy of a material using a DFT workflow on the Mat3ra platform.

<h2 style="color:green">Usage</h2>

1. Set material and calculation parameters in cell 1.2. below (or use the default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the result.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for material, workflow, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create material: materials are read from the `../uploads` folder — place files there manually or run a material creation notebook first. If the material is not found by name, Standata is used as a fallback. The material is then saved to the platform.
1. Create workflow and set its parameters: select application, load total energy workflow from Standata, optionally add relaxation or adjust model/method parameters, and save the workflow to the platform.
1. Configure compute: get list of clusters and create compute configuration with selected cluster, queue, and number of processors.
1. Create the job with material and workflow configuration: assemble the job from material, workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: get and display the properies.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install("mat3ra-utils")
    from mat3ra.utils.jupyterlite.packages import install_packages

    await install_packages("api_examples")


### 1.2. Set parameters and configurations for the workflow and job

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
# Set organization name to use it as the owner, otherwise your personal account is used
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"  # Name of the material to load from local file or Standata

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "total_energy.json"
APPLICATION_NAME = "espresso"  # Specify application name (e.g., "espresso", "vasp", "nwchem")
ADD_RELAXATION = False  # Whether to add relaxation subworkflow

MY_WORKFLOW_NAME = "Total Energy" + " (relax)" if ADD_RELAXATION else ""

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
JOB_NAME = f"{MY_WORKFLOW_NAME} {timestamp}"
POLL_INTERVAL = 30  # seconds

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from utils.auth import authenticate

# Authenticate in the browser and have credentials stored in environment variables
await authenticate()

### 2.2. Initialize API Client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load material from local file (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from utils.visualize import visualize_materials as visualize
from utils.jupyterlite import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from utils.api import get_or_create_material

saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)

## 4. Create workflow and set its parameters
### 4.1. Get list of applications and select one

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Create workflow from standard workflows and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from utils.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Modify workflow (Optional)
#### 4.3.1. Add relaxation

In [ ]:
if ADD_RELAXATION:
    workflow.add_relaxation()


#### 4.3.2. Modify model and method parameters (Optional)
Uncomment the code below and adjust selection of model parameters as needed.

In [ ]:
# # Example: Change model parameters
#
# from mat3ra.mode import Model
# from mat3ra.standata.model_tree import ModelTreeStandata
#
# model_config = ModelTreeStandata.get_model_by_parameters(
#     type="dft",
#     subtype="gga",
#     functional="pbe"
# )
# model_config["method"] = {"type": "pseudopotential", "subtype": "us"}
# model = Model.create(model_config)
#
# for subworkflow in workflow.subworkflows:
#     subworkflow.model = model
#
# if ADD_RELAXATION:
#     # If relaxation subworkflow is added, set the same model for it
#     workflow.relaxation_subworkflow.model = model
#
# # Preview modified workflow
# visualize_workflow(workflow)

#### 4.3.3. Modify important settings
Set k-grid.

In [ ]:
# from mat3ra.wode.context.providers import PointsGridDataProvider
# RELAXATION_KGRID = [2, 2, 2]
# SCF_KGRID = [4, 4, 4]
# new_context_relax = PointsGridDataProvider(dimensions=RELAXATION_KGRID, isEdited=True).yield_data() if ADD_RELAXATION else None
# new_context_scf = PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).yield_data()
#
# if ADD_RELAXATION:
#     relaxation_subworkflow = workflow.relaxation_subworkflow
#     unit_to_modify_relax = relaxation_subworkflow.get_unit_by_name(name_regex="relax")
#     unit_to_modify_relax.add_context(new_context_relax)
#     # Set the modified unit back to the workflow
#     # Option 1: direct set by unit object, replacing the existing one
#     relaxation_subworkflow.set_unit(unit_to_modify_relax)
#
# band_gap_subworkflow = workflow.subworkflows[1 if ADD_RELAXATION else 0]
# unit_to_modify_scf = band_gap_subworkflow.get_unit_by_name(name="pw_scf")
# unit_to_modify_scf.add_context(new_context_scf)
# band_gap_subworkflow.set_unit(unit_to_modify_scf)
#
# # Preview modified workflow
# visualize_workflow(workflow)

### 4.4. Save workflow to collection

In [ ]:
from utils.generic import dict_to_namespace
from utils.api import get_or_create_workflow

workflow_id_or_dict = None

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration for the job


In [ ]:
from mat3ra.ide.compute import Compute

# Select cluster: use specified name if provided, otherwise use first available
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job with material and workflow configuration
### 6.1. Create job

In [ ]:
from utils.api import create_job
from utils.visualize import display_JSON

print(f"Material: {saved_material.id}")
print(f"Workflow: {saved_workflow.id}")
print(f"Project: {project_id}")

job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=saved_workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=JOB_NAME,
    compute=compute.to_dict()
)

job = dict_to_namespace(job_response)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from utils.api import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve results

In [ ]:
from mat3ra.prode import PropertyName
from utils.visualize import visualize_properties

property_data = client.properties.get_for_job(job_id, property_name=PropertyName.scalar.total_energy.value)
visualize_properties(property_data, title="Total Energy")
